In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 08 — Reference coding solutions

These are instructor reference implementations, not evidence of your independent mastery. Run after attempting your own version. All examples use explicit small inputs and the declared dataset partitions.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
from itertools import combinations
import numpy as np
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/manifest.json').exists() and (p/'src/shape_lab').exists()),None)
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted course folder.')
sys.path.insert(0,str(ROOT/'src'))
from shape_lab.data import load_iris_data,load_digit_data,digit_example
from shape_lab.algebra import closure,validate_complex,rank_mod2,boundary_matrix,betti_numbers
from shape_lab.topology import component_labels
from shape_lab.persistence import ordered_filtration,rips_filtration,reduce_boundary,persistent_homology,diagram,pixel_filtration,finite_bottleneck
print('Working inside',ROOT.name,'— no network data downloads.')

Working inside course — no network data downloads.


## 08.C1 — Read a half-open barcode

Given pairs (birth,death), count intervals alive at t using birth<=t<death. Infinity is allowed.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

The left endpoint is included; the right endpoint is excluded.

</details>

In [3]:
def alive_count(pairs,t):
    return sum(b<=t<d for b,d in pairs)

In [4]:
pairs=[(0,1),(1,2),(0,np.inf)]
assert alive_count(pairs,0)==2 and alive_count(pairs,1)==2 and alive_count(pairs,2)==1
print("Endpoint convention checked at exact births and deaths")

Endpoint convention checked at exact births and deaths


**Why this works and what it does not establish:** Checking only thresholds between events can hide an endpoint-convention bug. Exact endpoints are essential in mathematical tests.

## 08.C2 — Summarize finite lifetimes

Return the sum of lifetimes of finite intervals. Do not replace infinity by an arbitrary large number.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Test finiteness of death before subtracting birth.

</details>

In [5]:
def finite_lifetime_sum(pairs):
    return sum(d-b for b,d in pairs if np.isfinite(d))

In [6]:
assert finite_lifetime_sum([(1,3),(0,np.inf),(.5,.5)])==2
assert finite_lifetime_sum([])==0
print("Essential interval excluded from finite lifetime sum")

Essential interval excluded from finite lifetime sum


**Why this works and what it does not establish:** Exclusion is a declared representation choice, not a statement that surviving components are unimportant. Preserve their information separately, such as through Betti counts.

## 08.C3 — Read the rank invariant

For s<=t, return the number of intervals born by s and surviving after t.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

A feature born after s cannot contribute to a map starting at s.

</details>

In [7]:
def barcode_rank(pairs,s,t):
    return sum(b<=s and t<d for b,d in pairs)

In [8]:
pairs=[(0,2),(1,3),(0,np.inf)]
assert barcode_rank(pairs,0,2)==1
assert barcode_rank(pairs,1,2)==2
assert barcode_rank(pairs,1,1)==3
print("Two-scale survival differs from counting each snapshot independently")

Two-scale survival differs from counting each snapshot independently


**Why this works and what it does not establish:** This counts the rank of the homology map under interval decomposition. It tests identity through the filtration, not only the number of classes present at a single threshold.

## 08.C4 — Recover the exact square interval

Compute the finite H1 diagram of the four unit-square corners using the core implementation. Return a two-column array.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Predict the result from side and diagonal lengths before calling any function.

</details>

In [9]:
def square_diagram():
    X=np.array([[0,0],[1,0],[1,1],[0,1]],float)
    return diagram(persistent_homology(rips_filtration(X)),1,finite_only=True)

In [10]:
D=square_diagram()
assert D.shape==(1,2) and np.allclose(D,[[1,np.sqrt(2)]])
print("Square H1 interval:",D)

Square H1 interval: [[1.         1.41421356]]


**Why this works and what it does not establish:** The loop appears when all four sides are present and dies when clique triangles fill it. Computing through triangles is enough to determine H1 deaths.

## 08.C5 — Inspect a real image persistence diagram

Return finite H1 birth/death pairs for a given supplied digit image, using f=1-intensity/16.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Use the same pixel object and entrance convention as the threshold lesson.

</details>

In [11]:
def digit_h1_diagram(image):
    f=pixel_filtration(1-np.asarray(image,dtype=float)/16)
    return diagram(persistent_homology(f),1,finite_only=True)

In [12]:
image,sample_id=digit_example(8);D=digit_h1_diagram(image)
assert D.shape[1]==2 and np.all(D[:,1]>D[:,0])
assert np.all(D>=0) and np.all(D<=1)
print("Training image",sample_id,"finite H1 diagram:",D)

Training image 8 finite H1 diagram: [[0.1875 0.9375]
 [0.375  1.    ]]


**Why this works and what it does not establish:** No H1 class is essential after every square of the rectangular image has entered. That justifies using the completed finite H1 diagram here, not removing infinite bars indiscriminately in every application.